In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [5]:

df = pd.read_csv('Dry_Bean_Dataset.csv')

print(df.shape)
print(df.head())
print(df.dtypes)

(13611, 17)
    Area  Perimeter  MajorAxisLength  MinorAxisLength  AspectRation  \
0  28395    610.291       208.178117       173.888747      1.197191   
1  28734    638.018       200.524796       182.734419      1.097356   
2  29380    624.110       212.826130       175.931143      1.209713   
3  30008    645.884       210.557999       182.516516      1.153638   
4  30140    620.134       201.847882       190.279279      1.060798   

   Eccentricity  ConvexArea  EquivDiameter    Extent  Solidity  roundness  \
0      0.549812       28715     190.141097  0.763923  0.988856   0.958027   
1      0.411785       29172     191.272751  0.783968  0.984986   0.887034   
2      0.562727       29690     193.410904  0.778113  0.989559   0.947849   
3      0.498616       30724     195.467062  0.782681  0.976696   0.903936   
4      0.333680       30417     195.896503  0.773098  0.990893   0.984877   

   Compactness  ShapeFactor1  ShapeFactor2  ShapeFactor3  ShapeFactor4  Class  
0     0.913358    

In [6]:
print(df.isnull().sum())       
print(df.duplicated().sum()) 

Area               0
Perimeter          0
MajorAxisLength    0
MinorAxisLength    0
AspectRation       0
Eccentricity       0
ConvexArea         0
EquivDiameter      0
Extent             0
Solidity           0
roundness          0
Compactness        0
ShapeFactor1       0
ShapeFactor2       0
ShapeFactor3       0
ShapeFactor4       0
Class              0
dtype: int64
68


In [7]:
le = LabelEncoder()
df['Class_encoded'] = le.fit_transform(df['Class'])

X = df.drop(columns=['Class', 'Class_encoded'])
y = df['Class_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y 
)

print(X_train.shape, X_test.shape)

(10888, 16) (2723, 16)


In [9]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

K = 22
knn_base = KNeighborsClassifier(n_neighbors=K)

knn_base.fit(X_train, y_train)
y_pred_base = knn_base.predict(X_test)

acc_base = accuracy_score(y_test, y_pred_base)
f1_base = f1_score(y_test, y_pred_base, average='weighted')

print(f"Accuracy:      {acc_base:.4f}")
print(f"F1 (weighted): {f1_base:.4f}\n")
print(classification_report(y_test, y_pred_base))

Accuracy:      0.6908
F1 (weighted): 0.6819

              precision    recall  f1-score   support

           0       0.51      0.41      0.45       265
           1       1.00      1.00      1.00       104
           2       0.64      0.65      0.65       326
           3       0.78      0.90      0.84       709
           4       0.67      0.65      0.66       386
           5       0.74      0.44      0.55       406
           6       0.61      0.74      0.67       527

    accuracy                           0.69      2723
   macro avg       0.71      0.68      0.69      2723
weighted avg       0.69      0.69      0.68      2723



In [10]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold, KFold
import numpy as np

params = {
    'n_neighbors': list(range(1, 31)),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}


# Стратегия 1: StratifiedKFold

In [12]:
cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=params,
    cv=cv_stratified,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"F1 (weighted): {grid_search.best_score_:.4f}")


Fitting 5 folds for each of 120 candidates, totalling 600 fits
Лучшие параметры: {'metric': 'manhattan', 'n_neighbors': 4, 'weights': 'distance'}
F1 (weighted): 0.7956


## Стратегия 2: KFold

In [ ]:
cv_kfold = KFold(n_splits=10, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=KNeighborsClassifier(),
    param_distributions={
        'n_neighbors': list(range(1, 51)),
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan', 'minkowski']
    },
    n_iter=30,                  
    cv=cv_kfold,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
random_search.fit(X_train, y_train)

print(f"Лучшие параметры: {random_search.best_params_}")
print(f"F1 (weighted): {random_search.best_score_:.4f}")

Fitting 10 folds for each of 30 candidates, totalling 300 fits
Лучшие параметры: {'weights': 'distance', 'n_neighbors': 7, 'metric': 'manhattan'}
F1 (weighted): 0.8005


In [ ]:


best_grid   = grid_search.best_estimator_
best_random = random_search.best_estimator_

y_pred_grid   = best_grid.predict(X_test)
y_pred_random = best_random.predict(X_test)

results = {
    'Модель': [
        f'Базовая (K={K})',
        f'GridSearchCV (K={grid_search.best_params_["n_neighbors"]})',
        f'RandomizedSearchCV (K={random_search.best_params_["n_neighbors"]})'
    ],
    'Accuracy': [
        accuracy_score(y_test, y_pred_base),
        accuracy_score(y_test, y_pred_grid),
        accuracy_score(y_test, y_pred_random)
    ],
    'F1 (weighted)': [
        f1_score(y_test, y_pred_base,   average='weighted'),
        f1_score(y_test, y_pred_grid,   average='weighted'),
        f1_score(y_test, y_pred_random, average='weighted')
    ],
}

df_results = pd.DataFrame(results)
df_results = df_results.set_index('Модель').round(4)
print("\nСравнение моделей ")
print(df_results)



Сравнение моделей 
                          Accuracy  F1 (weighted)
Модель                                           
Базовая (K=22)              0.6908         0.6819
GridSearchCV (K=4)          0.8021         0.8011
RandomizedSearchCV (K=7)    0.8002         0.7986


# Вывод
##### Подбор гиперпараметра K с помощью GridSearchCV и RandomizedSearchCV дал прирост качества, по сравнению с базовой моделью (F1: 0.68 → 0.80). Базовое значение K=22 оказалось избыточным — модель усредняла слишком много соседей и теряла точность на схожих классах. Оба метода поиска пришли к близкому результату (K=4 и K=7), что подтверждает: оптимальное K лежит в малом диапазоне. Это наглядно демонстрирует, что подбор гиперпараметров через кросс-валидацию значительно эффективнее произвольного выбора.